# 🎮 Halo CE Xbox (Build 2276) — Cloud Muscle Setup (A100 GPU)

> **Target Profile:** NVIDIA A100 GPU | Standard Memory Shape (NO High-RAM)  
> **Upstream Repositories:**  
> - `kuna`: [https://github.com/Noelo-Lab/kuna](https://github.com/Noelo-Lab/kuna) (Embedded SLEIGH x86 decompiler)  
> - `rea`: [https://github.com/morluto/rea](https://github.com/morluto/rea) (Reverse Engineer Anything framework)  
> - `rea-kuna`: [https://github.com/nickarcade/rea-kuna](https://github.com/nickarcade/rea-kuna) (Kuna provider & CLI interop)  
> **Project Codebase:** [https://github.com/nickarcade/halo](https://github.com/nickarcade/halo) (`mainrevised2`)  

---
### 📌 Critical Rules & Directives
1. **Strictly NO `rtk` Prefix:** Execute all commands directly without `rtk`.
2. **Decompiled Corpus & `tools/extract_fn.py`:** **Never read or view `cachebeta.xbe` directly.** All decompiled code, disassembly, and neural embeddings are queried via `tools/extract_fn.py` against `halo_decompiled/`.
3. **Decompiler Interoperability:** Uses `kuna` as the high-speed SLEIGH decompiler engine for `rea` via the `kuna-rea-bridge` and `rea-kuna` provider.


## 1. Hardware & GPU Attestation (A100 GPU, Standard Memory)
Verifies that the instance is allocated on an NVIDIA A100 GPU with standard memory.

In [ ]:
!nvidia-smi
!free -h
!lscpu | grep 'Model name\|CPU(s):'

## 2. Core Dependencies & Toolchains
Installs the Rust toolchain (`rustc`, `cargo`), Node.js 22 LTS, and native build dependencies.

In [ ]:
# Install build essentials
!apt-get update -qq && apt-get install -y -qq build-essential cmake pkg-config libssl-dev git curl

# Install Rust toolchain
import os
if not os.path.exists('/root/.cargo/bin/cargo'):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ.get('PATH', '')}"
!rustc --version && cargo --version

# Install Node.js 22 LTS
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y -qq nodejs
!node -v && npm -v

## 3. Clone, Build & Install `kuna` from Source (`Noelo-Lab/kuna`)
Compiles the latest release of `kuna` with embedded SLEIGH specs for x86.

In [ ]:
import os
if not os.path.exists('/content/kuna'):
    !git clone --depth 1 https://github.com/Noelo-Lab/kuna.git /content/kuna
else:
    !cd /content/kuna && git fetch origin && git reset --hard origin/main || git reset --hard origin/master

# Build Kuna in release mode
!cd /content/kuna && cargo build --release

# Install globally to /usr/local/bin
!cp /content/kuna/target/release/kuna /usr/local/bin/kuna && chmod +x /usr/local/bin/kuna
!kuna --version

## 4. Clone, Build & Install `rea` and `rea-kuna`
Builds `morluto/rea` and `nickarcade/rea-kuna` with full TypeScript compilation.

In [ ]:
import os
if not os.path.exists('/content/rea'):
    !git clone --depth 1 https://github.com/morluto/rea.git /content/rea
else:
    !cd /content/rea && git fetch origin && git reset --hard origin/main

if not os.path.exists('/content/rea-kuna'):
    !git clone --depth 1 https://github.com/nickarcade/rea-kuna.git /content/rea-kuna
else:
    !cd /content/rea-kuna && git fetch origin && git reset --hard origin/main

# Build both packages
!cd /content/rea && npm ci && npm run build:cached
!cd /content/rea-kuna && npm ci && npm run build:cached

# Symlink globally
!ln -sf /content/rea/scripts/rea.mjs /usr/local/bin/rea
!ln -sf /content/rea-kuna/scripts/rea.mjs /usr/local/bin/rea-kuna
!chmod +x /usr/local/bin/rea /usr/local/bin/rea-kuna
!rea --version && rea-kuna --version

## 5. Kuna <-> REA Decompiler Interoperability Bridge
Because `rea` primarily targets Ghidra and IDA Pro for native binary decompilation, we install a dual-layer interoperability bridge:
1. **`rea-kuna` Commands**: Native commands exposed directly on the CLI:
   - `rea-kuna kuna-decompile <binary> <func_or_0xaddr> [--addr] [--json]`
   - `rea-kuna kuna-disassemble <binary> <func_or_0xaddr> [--addr] [--json]`
   - `rea-kuna kuna-xrefs <binary> --to <func> [--json]`
   - `rea-kuna kuna-functions <binary> --summary [--json]`
   - `rea-kuna kuna-crypto <binary> [--json]`
2. **`kuna-rea-bridge`**: A headless provider shim installed at `/usr/local/bin/kuna-rea-bridge` allowing scripts and tools to query decompiler output via unified CLI args.

In [ ]:
bridge_script = '''#!/usr/bin/env python3
import sys
import subprocess

if len(sys.argv) < 3:
    print('Usage: kuna-rea-bridge <binary> <func_or_0xaddr> [--disasm] [--json]')
    sys.exit(1)

binary = sys.argv[1]
target = sys.argv[2]
is_disasm = '--disasm' in sys.argv
is_json = '--json' in sys.argv

cmd = ['kuna', 'disassemble' if is_disasm else 'decompile', binary, target]
if target.startswith('0x') or target.isdigit():
    cmd.append('--addr')
if is_json:
    cmd.append('--json')

res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if res.returncode != 0:
    sys.stderr.write(res.stderr)
    sys.exit(res.returncode)

print(res.stdout)
'''

with open('/usr/local/bin/kuna-rea-bridge', 'w') as f:
    f.write(bridge_script)
!chmod +x /usr/local/bin/kuna-rea-bridge
!kuna-rea-bridge --help 2>&1 || true

## 6. Halo CE Decompiled Corpus & `tools/extract_fn.py`
> ⚠️ **CRITICAL DIRECTIVE:** Do NOT read or view `cachebeta.xbe` directly.
Decompiled code, linear disassembly, and neural embeddings are indexed in `halo_decompiled/` and queried through `tools/extract_fn.py`.

In [ ]:
import os
if os.path.exists('/content/tools/extract_fn.py'):
    !python3 /content/tools/extract_fn.py --info 0x1c4990
else:
    print('tools/extract_fn.py will be synchronized during project workflow.')

## 7. Diagnostics & Readiness Report
Verifies end-to-end functionality of Kuna, REA, and the decompiler bridge.

In [ ]:
print('=== Kuna ===')
!kuna --version
print('\n=== REA Doctor ===')
!rea doctor 2>/dev/null || true
print('\n=== Bridge ===')
!kuna-rea-bridge --help 2>&1 || true
print('\n✅ Cloud RE Server Ready for Disassembly & Decompilation Offloading!')